# Семинар 2. Сегментация клиентов методом кластеризации и расчет RFM

**Цель семинара:** Познакомиться с задачей поведенческой сегментации, научиться рассчитывать аналитические RFM-метрики (Recency, Frequency, Monetary) из сырого журнала транзакций `transactions.csv` и применять алгоритм K-Means для выделения клиентских сегментов (`Cluster_ID`). Собранный конвейер будет оформлен в промышленную функцию.

### 🔧 Настройка окружения и импорт библиотек

В этом семинаре нам понадобятся модули машинного обучения из `scikit-learn` для масштабирования и кластеризации.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)


---

## 📥 Шаг 1. Инициализация локального контура (Трек Б: Транзакции)

Модели машинного обучения не умеют работать с длинной историей действий напрямую. Главная задача этого этапа — агрегировать лог операций `transactions.csv` до одной строки на каждого уникального клиента (`Target_ID`).


In [ ]:
INPUT_DIR = os.path.join(".", "data", "input")
trans_csv_path = os.path.abspath(os.path.join(INPUT_DIR, "transactions.csv"))

if not os.path.exists(trans_csv_path):
    print(f"⚠️ Файл не найден: {trans_csv_path}")

print(f"Путь к логу транзакций инициализирован: {trans_csv_path}")


---

## 🛠 ЗАДАНИЕ 1: Очистка лога транзакций и обработка выбросов
**Бизнес-контекст:** Транзакционные системы часто фиксируют возвраты (отрицательные суммы) или содержат системные сбои (аномально огромные платежи). Перед расчетом ценности клиента (Monetary) мы обязаны отфильтровать этот шум, иначе VIP-сегмент будет сформирован из ошибочных данных.

**Инструкция (TODO):**
1. Загрузите файл в датафрейм `df_trans`.
2. Приведите колонку `Trans_Date` к формату datetime.
3. Оставьте в датасете только те строки, где `Trans_Amount > 0`.
4. Отфильтруйте экстремальные выбросы: ограничьте `Trans_Amount` 99-м перцентилем (оставьте строки `Trans_Amount <= quantile(0.99)`).

*🤖 Теги для AI-ментора: `#SEM2_TASK1_START`, `#SEM2_TASK1_BUG`, `#SEM2_TASK1_WHY`*


In [ ]:
# [MASTER SOLUTION]
df_trans = pd.read_csv(trans_csv_path)

# 1. Приведение дат
df_trans['Trans_Date'] = pd.to_datetime(df_trans['Trans_Date'])

# 2. Очистка от отрицательных сумм и нулей
initial_shape = df_trans.shape
df_trans = df_trans[df_trans['Trans_Amount'] > 0]

# 3. Устранение экстремальных выбросов (99-й квантиль)
q99 = df_trans['Trans_Amount'].quantile(0.99)
df_trans = df_trans[df_trans['Trans_Amount'] <= q99]

print(f"Очистка завершена. Исходный размер: {initial_shape}, Текущий: {df_trans.shape}")
print(f"Порог 99% отсечения составил: {q99:.2f}")


In [ ]:
# [STUDENT TEMPLATE]
# TODO: 1.1. Считайте транзакции и переведите Trans_Date в формат pd.to_datetime
# TODO: 1.2. Отфильтруйте df_trans, оставив только суммы (Trans_Amount) строго больше нуля
# TODO: 1.3. Рассчитайте 99-й перцентиль (.quantile(0.99)) для колонки сумм
# TODO: 1.4. Отфильтруйте df_trans, оставив суммы меньше либо равные рассчитанному квантилю
raise NotImplementedError("Задание 1 не выполнено! Удалите эту строку и напишите свой код.")

df_trans = pd.read_csv(...)
df_trans['Trans_Date'] = ...

# Фильтрация бизнес-аномалий
df_trans = ...

# Обработка выбросов
q99 = ...
df_trans = ...

print(f"Очистка завершена. Текущий размер: {df_trans.shape}")


---

## 🛠 ЗАДАНИЕ 2: Агрегация лога до уровня клиента (Расчет RFM)
**Бизнес-контекст:** Чтобы модель предсказывала поведение клиента, мы сжимаем его историю до 3 метрик:
* **Recency (Давность):** Сколько дней прошло с последней транзакции. Уходящие клиенты перестают покупать заранее (рост Recency).
* **Frequency (Частота):** Общее количество операций. Индикатор лояльности.
* **Monetary (Ценность):** Сумма всех трат. 

**Инструкция (TODO):**
1. Определите `snapshot_date` (дата создания отчета) как максимальная дата транзакции во всем датасете + 1 день.
2. Используя `.groupby('Target_ID').agg(...)`, рассчитайте 3 метрики:
   - Для давности: `Trans_Date: lambda x: (snapshot_date - x.max()).days`
   - Для частоты: `Transaction_ID: 'count'` (если нет ID, можно считать по дате)
   - Для ценности: `Trans_Amount: 'sum'`
3. Переименуйте колонки агрегированного датасета в `Recency`, `Frequency`, `Monetary`.

*🤖 Теги для AI-ментора: `#SEM2_TASK2_START`, `#SEM2_TASK2_BUG`*


In [ ]:
# [MASTER SOLUTION]
# Точка отсчета для расчета давности (имитация "сегодня")
snapshot_date = df_trans['Trans_Date'].max() + pd.Timedelta(days=1)

# Агрегация RFM
df_rfm = df_trans.groupby('Target_ID').agg({
    'Trans_Date': lambda x: (snapshot_date - x.max()).days,
    'Transaction_ID': 'count',
    'Trans_Amount': 'sum'
}).reset_index()

# Переименование колонок для удобства
df_rfm.rename(columns={
    'Trans_Date': 'Recency',
    'Transaction_ID': 'Frequency',
    'Trans_Amount': 'Monetary'
}, inplace=True)

display(df_rfm.head())


In [ ]:
# [STUDENT TEMPLATE]
# TODO: 2.1. Рассчитайте snapshot_date: максимальная Trans_Date во всем df_trans + pd.Timedelta(days=1)
# TODO: 2.2. Сгруппируйте df_trans по 'Target_ID' и примените метод .agg()
# TODO: 2.3. В словаре .agg укажите правила: для дат - анонимная функция расчета дней, для ID - count, для сумм - sum
# TODO: 2.4. Вызовите .reset_index() после группировки и переименуйте столбцы в Recency, Frequency, Monetary
raise NotImplementedError("Задание 2 не выполнено! Удалите эту строку и напишите свой код.")

snapshot_date = ...

df_rfm = df_trans.groupby(...).agg({
    'Trans_Date': lambda x: (snapshot_date - ...).days,
    'Transaction_ID': ...,
    'Trans_Amount': ...
}).reset_index()

df_rfm.rename(columns={
    # Словарь переименований
}, inplace=True)

display(df_rfm.head())


---

## 🛠 ЗАДАНИЕ 3: Масштабирование метрик (StandardScaler)
**Бизнес-контекст:** Алгоритм кластеризации K-Means измеряет "расстояние" между клиентами (Евклидова метрика). Если `Monetary` исчисляется в тысячах (10000 руб.), а `Frequency` в единицах (5 покупок), алгоритм сочтет деньги бесконечно более важным признаком, проигнорировав частоту. Все оси должны быть приведены к единому масштабу ($mean=0$, $std=1$).

**Инструкция (TODO):**
1. Инициализируйте объект `StandardScaler()`.
2. Обучите скалер и преобразуйте колонки `['Recency', 'Frequency', 'Monetary']` с помощью `.fit_transform()`.
3. Сохраните результат в новый датафрейм `df_rfm_scaled`.

*🤖 Теги для AI-ментора: `#SEM2_TASK3_START`, `#SEM2_TASK3_WHY`*


In [ ]:
# [MASTER SOLUTION]
scaler = StandardScaler()
rfm_features = ['Recency', 'Frequency', 'Monetary']

# Обучение и трансформация
scaled_data = scaler.fit_transform(df_rfm[rfm_features])

# Упаковка обратно в DataFrame для удобства
df_rfm_scaled = pd.DataFrame(scaled_data, columns=rfm_features, index=df_rfm.index)

print("Среднее после масштабирования (должно быть ~0):", df_rfm_scaled['Monetary'].mean())
print("Ст. отклонение (должно быть ~1):", df_rfm_scaled['Monetary'].std())


In [ ]:
# [STUDENT TEMPLATE]
# TODO: 3.1. Создайте экземпляр класса StandardScaler()
# TODO: 3.2. Примените метод .fit_transform() только к 3 числовым колонкам датафрейма df_rfm
# TODO: 3.3. Создайте df_rfm_scaled на основе полученной матрицы
raise NotImplementedError("Задание 3 не выполнено! Удалите эту строку и напишите свой код.")

scaler = ...
rfm_features = ['Recency', 'Frequency', 'Monetary']

scaled_data = scaler....(df_rfm[rfm_features])
df_rfm_scaled = pd.DataFrame(scaled_data, columns=rfm_features, index=df_rfm.index)


---

## 🛠 ЗАДАНИЕ 4: Поиск оптимального числа сегментов (K-Means)
**Бизнес-контекст:** Мы не знаем заранее, сколько поведенческих групп (сегментов) есть в базе. 3? 5? 10? Мы используем "Метод локтя" (Inertia - сумма квадратов расстояний внутри кластеров) и коэффициент силуэта (Silhouette - насколько объекты внутри кластера похожи друг на друга по сравнению с другими кластерами) для поиска оптимального `K`.

**Инструкция (TODO):**
1. Напишите цикл `for k in range(2, 9):`.
2. Внутри инициализируйте `KMeans(n_clusters=k, random_state=42)`.
3. Обучите модель (`.fit()`) на масштабированных данных `df_rfm_scaled`.
4. Сохраните модельную метрику `.inertia_` в список.
5. Рассчитайте `silhouette_score(данные, метки)` и тоже сохраните.


In [ ]:
# [MASTER SOLUTION]
inertia = []
silhouette_scores = []
k_range = range(2, 9)

for k in k_range:
    # Защита от ошибки, если клиентов меньше, чем кластеров (для mock-данных)
    if k >= len(df_rfm_scaled):
        break
        
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(df_rfm_scaled)
    
    inertia.append(kmeans.inertia_)
    sil_score = silhouette_score(df_rfm_scaled, kmeans.labels_)
    silhouette_scores.append(sil_score)

# Визуализация (Для анализа студентом)
if inertia:
    fig, ax1 = plt.subplots(figsize=(10, 4))
    ax1.plot(range(2, 2+len(inertia)), inertia, marker='o', color='b', label='Метод Локтя (Inertia)')
    ax1.set_xlabel('Количество кластеров (k)')
    ax1.set_ylabel('Inertia', color='b')
    
    ax2 = ax1.twinx()
    ax2.plot(range(2, 2+len(silhouette_scores)), silhouette_scores, marker='x', color='r', label='Силуэт (Silhouette)')
    ax2.set_ylabel('Silhouette Score', color='r')
    
    plt.title('Определение оптимального числа сегментов')
    plt.show()


In [ ]:
# [STUDENT TEMPLATE]
# TODO: 4.1. Внутри цикла создайте KMeans с n_clusters=k и random_state=42
# TODO: 4.2. Вызовите .fit() на df_rfm_scaled
# TODO: 4.3. Добавьте значение inertia_ в список inertia
# TODO: 4.4. Добавьте silhouette_score в список silhouette_scores
raise NotImplementedError("Задание 4 не выполнено! Удалите эту строку и напишите свой код.")

inertia = []
silhouette_scores = []
k_range = range(2, 9)

for k in k_range:
    if k >= len(df_rfm_scaled):
        break
        
    kmeans = ...
    kmeans...
    
    inertia.append(...)
    silhouette_scores.append(silhouette_score(..., ...))

# Визуализация будет работать автоматически после заполнения списков


---

## 🛠 ЗАДАНИЕ 5: Финальная кластеризация и бизнес-маппинг
**Бизнес-контекст:** Определив оптимальное число (обычно 3 или 4), мы проводим финальную разметку базы. Каждому клиенту будет присвоен номер сегмента от `0` до `K-1`. В дальнейшем BI-аналитик переведет эти цифры в метки "VIP", "Спящие", "Группа риска".

**Инструкция (TODO):**
1. Инициализируйте `KMeans(n_clusters=3, random_state=42)`.
2. Обучите модель на отмасштабированных данных и получите метки `kmeans.labels_`.
3. Запишите эти метки в исходный (не масштабированный!) датафрейм `df_rfm` в колонку `Cluster_ID`.


In [ ]:
# [MASTER SOLUTION]
n_optimal_clusters = min(3, len(df_rfm_scaled)) # Динамически для mock-данных
final_kmeans = KMeans(n_clusters=n_optimal_clusters, random_state=42, n_init=10)
final_kmeans.fit(df_rfm_scaled)

# Присваиваем лейблы исходному агрегату
df_rfm['Cluster_ID'] = final_kmeans.labels_

print("Средние значения RFM по выделенным кластерам:")
display(df_rfm.groupby('Cluster_ID')[['Recency', 'Frequency', 'Monetary']].mean())


In [ ]:
# [STUDENT TEMPLATE]
# TODO: 5.1. Обучите финальную модель с оптимальным числом кластеров (например, 3)
# TODO: 5.2. Сохраните метки (labels_) как новую колонку 'Cluster_ID' в датафрейм df_rfm
raise NotImplementedError("Задание 5 не выполнено! Удалите эту строку и напишите свой код.")

final_kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
# Ваш код обучения...

df_rfm['Cluster_ID'] = ...

display(df_rfm.groupby('Cluster_ID')[['Recency', 'Frequency', 'Monetary']].mean())


---

## 🏗 ФИНАЛЬНАЯ СБОРКА: Сквозная функция compute_rfm_clusters

Соберем шаги обработки сырого лога, расчет RFM и кластеризацию в единую параметризованную production-функцию `compute_rfm_clusters`. Она станет частью модуля ядра `src/data_pipeline.py`. Это позволит в будущем гибко менять имена колонок, если структура корпоративной БД изменится.


In [ ]:
# [MASTER SOLUTION]
def compute_rfm_clusters(df_trans: pd.DataFrame, customer_id_col: str, date_col: str, amount_col: str, n_clusters: int = 3) -> pd.DataFrame:
    """
    Агрегация транзакционной истории, расчет поведенческих RFM-метрик
    и автоматическая кластеризация K-Means.
    """
    df = df_trans.copy()
    
    # 1. Очистка и подготовка
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    df = df[df[amount_col] > 0]
    
    q99 = df[amount_col].quantile(0.99)
    df = df[df[amount_col] <= q99]
    
    if df.empty:
        return pd.DataFrame()
        
    # 2. Агрегация RFM
    snapshot_date = df[date_col].max() + pd.Timedelta(days=1)
    
    df_rfm = df.groupby(customer_id_col).agg({
        date_col: lambda x: (snapshot_date - x.max()).days,
        customer_id_col: 'count', # Считаем количество записей
        amount_col: 'sum'
    }).rename(columns={
        customer_id_col: 'Frequency' # Временное имя, если столбец совпадает с индексом
    })
    
    # Сброс индекса, чтобы Target_ID снова стал колонкой
    df_rfm = df_rfm.rename_axis(customer_id_col).reset_index()
    
    # Переименование в бизнес-термины
    df_rfm.rename(columns={
        date_col: 'Recency',
        amount_col: 'Monetary'
    }, inplace=True)
    
    # Если клиентов меньше чем кластеров (защита)
    actual_clusters = min(n_clusters, len(df_rfm))
    
    if actual_clusters < 2:
        df_rfm['Cluster_ID'] = 0
        return df_rfm
        
    # 3. Масштабирование и Кластеризация
    scaler = StandardScaler()
    rfm_scaled = scaler.fit_transform(df_rfm[['Recency', 'Frequency', 'Monetary']])
    
    kmeans = KMeans(n_clusters=actual_clusters, random_state=42, n_init=10)
    df_rfm['Cluster_ID'] = kmeans.fit_predict(rfm_scaled)
    
    return df_rfm

# Тестирование функции
df_rfm_final = compute_rfm_clusters(
    df_trans=df_raw_trans if 'df_raw_trans' in locals() else pd.read_csv(trans_csv_path),
    customer_id_col='Target_ID',
    date_col='Trans_Date',
    amount_col='Trans_Amount'
)

print(f"Пайплайн RFM запущен! Размерность агрегата: {df_rfm_final.shape}")
display(df_rfm_final.head(3))


In [ ]:
# [STUDENT TEMPLATE]
def compute_rfm_clusters(df_trans: pd.DataFrame, customer_id_col: str, date_col: str, amount_col: str, n_clusters: int = 3) -> pd.DataFrame:
    """
    Агрегирует журнал транзакций до клиента, считает RFM и присваивает Cluster_ID.
    """
    # TODO: Соберите архитектуру функции, используя переменные customer_id_col, date_col, amount_col
    # ВАЖНО: Замените жесткие строки ('Trans_Date', 'Target_ID') на переданные аргументы
    raise NotImplementedError("Финальная сборка функции не выполнена!")
    
    df = df_trans.copy()
    
    # 1. Очистка
    # ...
    
    # 2. Агрегация RFM
    # ...
    
    # 3. Масштабирование и Кластеризация
    # ...
    
    return pd.DataFrame()

# Применение
df_rfm_final = compute_rfm_clusters(pd.read_csv(trans_csv_path), 'Target_ID', 'Trans_Date', 'Trans_Amount')


---

## 🛠 Автоматизированная проверка качества (Autocheck)

Скрипт ниже проверяет архитектуру вашей агрегационной функции `compute_rfm_clusters` на соответствие стандартам пайплайна.


In [ ]:
def run_autocheck(df_to_check):
    print("🚀 Тестирование функции compute_rfm_clusters...\n" + "-"*45)
    validation_status = True
    
    if not isinstance(df_to_check, pd.DataFrame) or df_to_check.empty:
        print("❌ Ошибка: Возвращен пустой или невалидный DataFrame.")
        return False
        
    expected_cols = ['Target_ID', 'Recency', 'Frequency', 'Monetary', 'Cluster_ID']
    missing_cols = [c for c in expected_cols if c not in df_to_check.columns]
    
    # Проверка структуры столбцов
    if missing_cols:
        print(f"❌ Ошибка: В таблице отсутствуют обязательные столбцы: {missing_cols}")
        validation_status = False
    else:
        print("✅ Бизнес-метрики RFM и Cluster_ID успешно рассчитаны.")
        
    # Проверка гранулярности
    if 'Target_ID' in df_to_check.columns:
        if df_to_check['Target_ID'].duplicated().sum() > 0:
            print("❌ Ошибка: Гранулярность нарушена. Один клиент встречается несколько раз.")
            validation_status = False
        else:
            print("✅ Группировка (groupby) проведена верно, строго одна строка на клиента.")
            
    # Проверка кластеров
    if 'Cluster_ID' in df_to_check.columns:
        unique_clusters = df_to_check['Cluster_ID'].nunique()
        print(f"✅ Найдено сегментов (кластеров): {unique_clusters}")
        
    print("-" * 45)
    if validation_status:
        print("🎉 ПОЗДРАВЛЯЕМ! Модуль сегментации готов.")
        print("Перенесите эту функцию в файл course_project/src/data_pipeline.py!")
    else:
        print("⚠️ Обнаружены технические дефекты. Проверьте реализацию вашей финальной функции.")

run_autocheck(df_rfm_final)
